# How far can you trust a weather forecast?

DCV dashboard (v2.1, 34 plots, 6 tabs). Forecast error vs station observations for Kolkata, Visakhapatnam and Thiruvananthapuram.

Reads the same data folders as before and writes only `Forecast_Dashboard.html`.

**How to use:** put this notebook next to your data, run all cells top to bottom. Each plot is its own function returning `(figure, caption)`, so you can also `import` them from a `.py` file.
To preview a tab inline use `preview_tab(0)` … `preview_tab(5)`; to write the HTML use `build()`.

In [53]:
import os
import pathlib
import sys
import glob
import textwrap
import traceback
import webbrowser

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Settings

In [54]:
try:
    DATA_DIR = os.path.dirname(os.path.abspath(__file__))  # running as a .py script
except NameError:
    DATA_DIR = os.getcwd()  # running inside a notebook
OUT_HTML = os.path.join(DATA_DIR, "Forecast_Dashboard.html")

CITIES = ["Kolkata", "Visakhapatnam", "Thiruvananthapuram"]
LEADS = list(range(1, 8))
MONSOON = [6, 7, 8, 9]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
SEASONS = ["Monsoon (Jun-Sep)", "Other months"]
RAIN_WET_MM = 1.0
WIN_START = pd.Timestamp("2022-06-01")
WIN_END = pd.Timestamp("2025-08-31")
RAIN_START = pd.Timestamp("2024-01-20")
TEMP_THR = 2.0
RAIN_THR = 50.0
GRID_KM = {"Kolkata": 3.4, "Visakhapatnam": 4.7, "Thiruvananthapuram": 5.5}

## Palette

In [55]:
BG = "#05050B"
CARD = "#0E0E1A"
EDGE = "#1E1E33"
INK = "#F2F4FF"
MUTED = "#8891B0"
SOFT = "#B0B8D0"
GRID = "#14141F"
HI = "#FFB020"
CYAN = "#22D3EE"
PINK = "#EC4899"
LIME = "#A3E635"
VIOLET = "#A78BFA"
ROSE = "#F472B6"

CITY_STYLE = {
    "Kolkata":            {"color": CYAN, "dash": "solid", "symbol": "circle"},
    "Visakhapatnam":      {"color": PINK, "dash": "dash",  "symbol": "diamond"},
    "Thiruvananthapuram": {"color": LIME, "dash": "dot",   "symbol": "square"},
}
LEAD_COL = {1: "#E0F7FF", 2: "#A5E8F6", 3: "#5ED3EA", 4: CYAN,
            5: "#1EA0B8", 6: HI, 7: PINK}
NEON_SCALE = [[0, "#0B0B18"], [0.2, "#1E1B4B"], [0.5, "#7C3AED"],
              [0.75, CYAN], [0.9, LIME], [1, HI]]

## Helpers

In [56]:
def find(pattern):
    for root in (DATA_DIR, os.getcwd()):
        hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
        if hits:
            return hits[-1]
    raise FileNotFoundError(f"pattern not found: {pattern}")

In [57]:
def to_long(df, prefix, obs_col):
    parts = []
    for k in LEADS:
        p = df[["city", "date", "year", "month", obs_col]].copy()
        p["obs"] = p.pop(obs_col)
        p["fc"] = df[f"{prefix}{k}"].values
        p["lead"] = k
        parts.append(p)
    out = pd.concat(parts, ignore_index=True)
    out["err"] = out["fc"] - out["obs"]
    out["ae"] = out["err"].abs()
    out["season"] = np.where(out["month"].isin(MONSOON), SEASONS[0], SEASONS[1])
    return out.dropna(subset=["obs", "fc"]).reset_index(drop=True)

In [58]:
def csi_by_lead(df, thr=RAIN_WET_MM):
    out = []
    for k in LEADS:
        d = df[df["lead"] == k]
        ow = d["obs"] >= thr
        fw = d["fc"] >= thr
        hit = int((ow & fw).sum()); miss = int((ow & ~fw).sum()); fa = int((~ow & fw).sum())
        tot = hit + miss + fa
        out.append(100.0 * hit / tot if tot > 0 else np.nan)
    return out

In [59]:
def pod_sr_by_lead(df, thr=RAIN_WET_MM):
    rows = []
    for k in LEADS:
        d = df[df["lead"] == k]
        ow = d["obs"] >= thr; fw = d["fc"] >= thr
        hit = int((ow & fw).sum()); miss = int((ow & ~fw).sum()); fa = int((~ow & fw).sum())
        rows.append({"lead": k,
                     "pod": hit / max(hit + miss, 1),
                     "sr":  hit / max(hit + fa, 1),
                     "hit": hit, "miss": miss, "fa": fa})
    return pd.DataFrame(rows).set_index("lead")

In [60]:
def trust_days(values, thr, higher_better=False):
    for i, v in enumerate(values):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        ok = (v >= thr) if higher_better else (v <= thr)
        if not ok:
            if i == 0:
                return 0.0
            a, b = values[i - 1], v
            if a == b:
                return float(i)
            return i + (a - thr) / (a - b) * (-1 if higher_better else 1)
    return 7.0

In [61]:
def rgba(hex_, a):
    h = hex_.lstrip("#")
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{a})"

In [62]:
def wrap(s, n=60):
    return "<br>".join(textwrap.wrap(str(s), n))

## Load Data

In [63]:
temp_wide = pd.read_parquet(find("*noaa_openmeteo_temperature_*.parquet"))
rain_wide = pd.read_parquet(find("*noaa_openmeteo_rain_*.parquet"))
temp_wide["date"] = pd.to_datetime(temp_wide["date"])
rain_wide["date"] = pd.to_datetime(rain_wide["date"])
T = to_long(temp_wide, "fc_tmax_lead", "TMAX")
R = to_long(rain_wide, "fc_rain_lead", "PRCP")

try:
    CLEAN_LOG = pd.read_csv(find("*cleaning_log_*.csv"))
    CLEAN_LOG["rows_before"] = pd.to_numeric(CLEAN_LOG["rows_before"], errors="coerce")
    CLEAN_LOG["rows_after"]  = pd.to_numeric(CLEAN_LOG["rows_after"], errors="coerce")
except Exception:
    CLEAN_LOG = None

try:
    INTERIM = pd.read_parquet(find("*noaa_ghcnd_3stations_*.parquet"))
    INTERIM["date"] = pd.to_datetime(INTERIM["date"])
    INTERIM_OK = True
except Exception:
    INTERIM = None
    INTERIM_OK = False

mae_pooled = T.groupby("lead")["ae"].mean().reindex(LEADS).tolist()
csi_pooled = csi_by_lead(R)
trust_t = trust_days(mae_pooled, TEMP_THR)
trust_r = trust_days(csi_pooled, RAIN_THR, higher_better=True)
trust_all = min(trust_t, trust_r)

## Theme

In [64]:
def theme(fig, title, xt=None, yt=None, h=460, legend=True, accent=HI):
    fig.update_layout(
        title=dict(text=title, x=0.0, xanchor="left", y=0.965, yanchor="top",
                   font=dict(size=15, color=INK, family="Space Grotesk, Inter, sans-serif")),
        height=h,
        font=dict(family="Inter, sans-serif", size=11, color=INK),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=64, r=22, t=80, b=74 if legend else 50),
        hoverlabel=dict(bgcolor=CARD, bordercolor=accent,
                        font=dict(family="JetBrains Mono, Inter, monospace",
                                  color=INK, size=12)),
        legend=dict(orientation="h", x=0, y=-0.2, yanchor="top",
                    font=dict(size=11, color=INK), bgcolor="rgba(0,0,0,0)"),
        showlegend=legend,
        colorway=[CYAN, PINK, LIME, VIOLET, HI, ROSE])
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                     tickfont=dict(color=MUTED), title_font=dict(color=MUTED, size=12))
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                     tickfont=dict(color=MUTED), title_font=dict(color=MUTED, size=12))
    if xt: fig.update_xaxes(title_text=xt)
    if yt: fig.update_yaxes(title_text=yt)
    return fig

## Tab 1 — Pulse (5)

In [65]:
def p_trust_bars():
    rows = []
    for city in CITIES:
        mae_c = T[T["city"] == city].groupby("lead")["ae"].mean().reindex(LEADS).tolist()
        csi_c = csi_by_lead(R[R["city"] == city])
        tt = trust_days(mae_c, TEMP_THR)
        tr = trust_days(csi_c, RAIN_THR, higher_better=True)
        rows.append(dict(city=city, temp=tt, rain=tr, overall=min(tt, tr)))
    df = pd.DataFrame(rows).set_index("city").reindex(CITIES)
    fig = go.Figure()
    for metric, label, col in [("temp", "Temperature", CYAN),
                               ("rain", "Rain", PINK),
                               ("overall", "Overall", HI)]:
        fig.add_trace(go.Bar(
            y=CITIES, x=df[metric].values, orientation="h", name=label,
            marker=dict(color=col, line=dict(color=BG, width=1)),
            text=[f"{v:.1f}d" for v in df[metric].values], textposition="outside",
            textfont=dict(color=col, size=12, family="JetBrains Mono"),
            hovertemplate=f"<b>%{{y}}</b><br>{label}: %{{x:.1f}} days<extra></extra>"))
    fig.update_layout(barmode="group", bargap=0.28)
    fig.update_xaxes(range=[0, 8.6], tickmode="array",
                     tickvals=list(range(8)), ticktext=[f"{i}d" for i in range(8)])
    fig.update_yaxes(autorange="reversed")
    theme(fig, "Trust horizon — temperature, rain, overall per city",
          "Days ahead the forecast stays within your standard", None)
    cap = (f"Each bar is how many days ahead the forecast still meets the standard "
           f"(temp error ≤ {TEMP_THR:.0f} °C, rain CSI ≥ {RAIN_THR:.0f}%). Overall is the shorter of the two — "
           f"a plan is only as safe as its weaker part. Interpolated between whole lead days.")
    return fig, cap

In [66]:
def p_cone():
    g = T.groupby("lead")["err"].agg(
        mean="mean",
        q10=lambda s: s.quantile(0.10), q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75), q90=lambda s: s.quantile(0.90),
    ).reindex(LEADS)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=LEADS + LEADS[::-1], y=g["q90"].tolist() + g["q10"].tolist()[::-1],
        fill="toself", fillcolor=rgba(CYAN, 0.10), line=dict(width=0),
        name="10–90% range", hoverinfo="skip"))
    fig.add_trace(go.Scatter(
        x=LEADS + LEADS[::-1], y=g["q75"].tolist() + g["q25"].tolist()[::-1],
        fill="toself", fillcolor=rgba(CYAN, 0.28), line=dict(width=0),
        name="25–75% range", hoverinfo="skip"))
    fig.add_trace(go.Scatter(
        x=LEADS, y=g["mean"], mode="lines+markers",
        line=dict(color=CYAN, width=3.5),
        marker=dict(size=11, color=CYAN, line=dict(color=BG, width=2)),
        name="Mean error",
        hovertemplate="<b>Lead %{x}</b><br>Mean: %{y:+.2f} °C<extra></extra>"))
    fig.add_hline(y=0, line_color=INK, line_dash="dot", line_width=1, opacity=0.4)
    fig.update_xaxes(tickmode="array", tickvals=LEADS, ticktext=[f"{k}d" for k in LEADS])
    theme(fig, "Cone of uncertainty — signed error widens with lead",
          "Lead time", "Error = forecast − station (°C)")
    cap = (f"Pale band = 10–90% of signed error, darker = 25–75%, cyan line = mean. "
           f"Day 1 sits tight ({g.loc[1,'q10']:+.1f} to {g.loc[1,'q90']:+.1f} °C); day 7 is wide "
           f"({g.loc[7,'q10']:+.1f} to {g.loc[7,'q90']:+.1f} °C). About {int((T['lead']==1).sum()):,} matched days per lead.")
    return fig, cap

In [67]:
def p_scorecard():
    within2 = [100 * (T.loc[T["lead"] == k, "ae"] <= TEMP_THR).mean() for k in LEADS]
    csi = csi_by_lead(R)
    pod = []
    for k in LEADS:
        d = R[R["lead"] == k]
        ow = d["obs"] >= RAIN_WET_MM; fw = d["fc"] >= RAIN_WET_MM
        hit = int((ow & fw).sum()); miss = int((ow & ~fw).sum())
        pod.append(100 * hit / max(hit + miss, 1))
    z = [within2, csi, pod]
    txt = [[("" if np.isnan(v) else f"{v:.0f}") for v in row] for row in z]
    fig = go.Figure(go.Heatmap(
        z=z, x=[f"{k}d" for k in LEADS],
        y=[f"Temp ≤ {TEMP_THR:.0f} °C", "Rain CSI", "Rain POD"],
        colorscale=NEON_SCALE, zmin=0, zmax=100,
        text=txt, texttemplate="%{text}",
        textfont=dict(size=13, family="JetBrains Mono", color=INK),
        xgap=3, ygap=3,
        colorbar=dict(title=dict(text="%", font=dict(size=10, color=MUTED)), thickness=10),
        hovertemplate="%{y}<br>%{x}: %{z:.0f}%<extra></extra>"))
    fig.update_yaxes(autorange="reversed", tickfont=dict(color=INK))
    theme(fig, "Scorecard — 3 honest measures, day 1 to day 7", legend=False)
    cap = (f"Rows: % of days within {TEMP_THR:.0f} °C, % of rain events forecast correctly (CSI), "
           f"% of wet days warned in advance (POD). Each falls left→right; temperature drops "
           f"{within2[0]-within2[6]:.0f} pts, rain CSI drops {csi[0]-csi[6]:.0f} pts.")
    return fig, cap

In [68]:
def p_rolling():
    idx = pd.date_range(WIN_START, WIN_END)
    fig = go.Figure()
    fig.update_xaxes(range=[str(WIN_START.date()), str(WIN_END.date())], type="date")
    for yr in range(WIN_START.year, WIN_END.year + 1):
        x0 = max(pd.Timestamp(f"{yr}-06-01"), WIN_START)
        x1 = min(pd.Timestamp(f"{yr}-09-30"), WIN_END)
        if x0 < x1:
            fig.add_vrect(x0=x0, x1=x1, fillcolor=LIME, opacity=0.055, line_width=0)
    for k, col, w in ((1, LEAD_COL[1], 1.8), (4, CYAN, 2.4), (7, PINK, 3.2)):
        s = T[T["lead"] == k].groupby("date")["ae"].mean().reindex(idx)
        roll = s.rolling(30, min_periods=15).mean().iloc[29:]
        fig.add_trace(go.Scatter(
            x=roll.index, y=roll.values, mode="lines", name=f"Lead {k}",
            line=dict(color=col, width=w),
            hovertemplate=f"<b>Lead {k}</b><br>%{{x|%d %b %Y}}<br>MAE: %{{y:.2f}} °C<extra></extra>"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                             name="Monsoon (Jun–Sep)",
                             marker=dict(symbol="square", size=11, color=LIME, opacity=0.3)))
    theme(fig, "30-day rolling MAE — error pulses up during each monsoon",
          "Valid date", "Rolling MAE (°C)")
    cap = ("30-day rolling mean absolute error, averaged over the 3 cities. Shaded bands are June–September. "
           "The day-7 curve (pink) repeatedly peaks in monsoon months; day 1 (pale cyan) is comparatively stable.")
    return fig, cap

In [69]:
def p_treemap():
    counts = T.groupby(["city", "season"]).size().reset_index(name="n")
    total = int(counts["n"].sum())
    labels = ["All matched days"]; parents = [""]; values = [total]; colors = [CARD]
    for city in CITIES:
        sub = counts[counts["city"] == city]
        if sub.empty:
            continue
        labels.append(city); parents.append("All matched days")
        values.append(int(sub["n"].sum())); colors.append(CITY_STYLE[city]["color"])
        for _, row in sub.iterrows():
            season_short = row["season"].split(" (")[0]
            labels.append(f"{city} · {season_short}")
            parents.append(city); values.append(int(row["n"]))
            colors.append(CITY_STYLE[city]["color"] if row["season"].startswith("Monsoon") else EDGE)
    fig = go.Figure(go.Treemap(
        labels=labels, parents=parents, values=values, branchvalues="total",
        marker=dict(colors=colors, line=dict(color=BG, width=3)),
        textinfo="label+value",
        textfont=dict(family="Inter", size=13, color=INK),
        hovertemplate="<b>%{label}</b><br>%{value:,} matched days<extra></extra>"))
    theme(fig, f"Sample sizes — {total:,} matched forecast-observation days", legend=False)
    biggest = counts.groupby("city")["n"].sum().idxmax()
    cap = (f"Each city's block splits into monsoon (brighter) and other months. Total matched days: {total:,}. "
           f"The biggest chunk is {biggest} ({int(counts.groupby('city')['n'].sum().max()):,} days).")
    return fig, cap

## Tab 2 — Temperature (6)

In [70]:
def p_ridge():
    lim = 10.0
    fig = go.Figure()
    for k in LEADS[::-1]:
        d = T.loc[T["lead"] == k, "err"].clip(-lim, lim)
        col = LEAD_COL[k]
        fig.add_trace(go.Violin(
            x=d, y0=f"Day {k}", orientation="h", side="positive", width=2.4,
            points=False, spanmode="hard", span=[-lim, lim],
            line=dict(color=col, width=2.2 if k == 7 else 1.4),
            fillcolor=rgba(col, 0.42 if k == 7 else 0.22),
            name=f"Lead {k}", showlegend=False,
            hovertemplate=f"<b>Lead {k}</b><br>%{{x:.1f}} °C<extra></extra>"))
    fig.add_vline(x=0, line_color=INK, line_dash="dash", line_width=1.2, opacity=0.8)
    fig.update_yaxes(categoryorder="array",
                     categoryarray=[f"Day {k}" for k in LEADS[::-1]],
                     tickfont=dict(color=INK, size=12))
    fig.update_xaxes(range=[-lim, lim], ticksuffix=" °C")
    theme(fig, "Error ridge — distributions widen and skew with lead time",
          "Forecast error (°C)", None, legend=False)
    sd1 = float(T.loc[T["lead"] == 1, "err"].std())
    sd7 = float(T.loc[T["lead"] == 7, "err"].std())
    cap = (f"Each ridge is signed error at one lead. Day-7 (pink) is {sd7/sd1:.1f}× wider than day 1 "
           f"(SD {sd1:.2f} → {sd7:.2f} °C). Bias stays small; growth is almost all scatter.")
    return fig, cap

In [71]:
def p_hexbin():
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Lead 1 day", "Lead 7 days"),
                        horizontal_spacing=0.07, shared_yaxes=True)
    lo = float(T["obs"].min()) - 1
    hi = float(T["obs"].max()) + 1
    corr = {}
    for j, k in enumerate((1, 7), 1):
        d = T[T["lead"] == k]
        corr[k] = float(d["obs"].corr(d["fc"]))
        kw = dict(x=d["obs"], y=d["fc"], nbinsx=28, nbinsy=28,
                  colorscale=NEON_SCALE, showscale=(j == 2),
                  hovertemplate="Station %{x:.1f} °C<br>Forecast %{y:.1f} °C<br>%{z} days<extra></extra>")
        if j == 2:
            kw["colorbar"] = dict(title=dict(text="Days", font=dict(size=10, color=MUTED)), thickness=10)
        fig.add_trace(go.Histogram2d(**kw), row=1, col=j)
        fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", showlegend=False,
                                 line=dict(color=INK, dash="dash", width=1.2), hoverinfo="skip"),
                      row=1, col=j)
    fig.update_xaxes(range=[lo, hi], ticksuffix="°C")
    fig.update_yaxes(range=[lo, hi], ticksuffix="°C")
    fig.update_xaxes(title_text="Station TMAX (°C)")
    fig.update_yaxes(title_text="Forecast TMAX (°C)", col=1)
    theme(fig, f"Where does the forecast land on the 1:1 line? (r = {corr[1]:.2f} → {corr[7]:.2f})", legend=False)
    cap = (f"Brighter cells hold more days; dashed = perfect forecast. Day 1 hugs it (r = {corr[1]:.2f}); "
           f"by day 7 the cloud spreads (r = {corr[7]:.2f}). Correlation can't see a constant offset — "
           f"that's the bias plot.")
    return fig, cap

In [72]:
def p_within_tol():
    xs = np.linspace(0, 8, 161)
    fig = go.Figure()
    for k in LEADS:
        a = np.sort(T.loc[T["lead"] == k, "ae"].values)
        y = 100 * np.searchsorted(a, xs, side="right") / max(len(a), 1)
        fig.add_trace(go.Scatter(
            x=xs, y=y, mode="lines", name=f"{k}d",
            line=dict(color=LEAD_COL[k], width=4 if k == 7 else 2),
            hovertemplate=f"Lead {k}<br>within %{{x:.1f}} °C: %{{y:.0f}}%<extra></extra>"))
    fig.add_vline(x=TEMP_THR, line_dash="dash", line_color=INK, line_width=1.2)
    theme(fig, "How often is it good enough? Cumulative share within x °C",
          "Absolute error (°C)", "Days with error ≤ x (%)")
    w1 = 100 * (T.loc[T["lead"] == 1, "ae"] <= TEMP_THR).mean()
    w7 = 100 * (T.loc[T["lead"] == 7, "ae"] <= TEMP_THR).mean()
    cap = (f"Each curve is one lead (light = short, pink = day 7). Where it crosses the dashed "
           f"{TEMP_THR:.0f} °C line gives the share of days that were close enough — {w1:.0f}% at day 1, "
           f"{w7:.0f}% at day 7.")
    return fig, cap

In [73]:
def p_slope():
    mae_c = T.groupby(["city", "lead"])["ae"].mean().unstack().reindex(CITIES)
    fig = go.Figure()
    for city in CITIES:
        st = CITY_STYLE[city]
        fig.add_trace(go.Scatter(
            x=LEADS, y=mae_c.loc[city], mode="lines+markers",
            name=city,
            line=dict(color=st["color"], width=2.6, dash=st["dash"]),
            marker=dict(size=9, symbol=st["symbol"], color=st["color"],
                        line=dict(color=BG, width=1.5)),
            hovertemplate=f"<b>{city}</b><br>Lead %{{x}}: %{{y:.2f}} °C<extra></extra>"))
    fig.update_xaxes(tickmode="array", tickvals=LEADS, ticktext=[f"{k}d" for k in LEADS])
    theme(fig, "Each city climbs at its own pace", "Lead time", "MAE (°C)")
    cap = ("At day 1 the three cities are close. By day 7 they diverge: "
           + ", ".join(f"{c} {mae_c.loc[c, 7]:.2f} °C" for c in CITIES)
           + " — the spread only opens up at long lead.")
    return fig, cap

In [74]:
def p_bias():
    bias_c = T.groupby(["city", "lead"])["err"].mean().unstack().reindex(CITIES)
    bias_pooled = T.groupby("lead")["err"].mean().reindex(LEADS)
    fig = go.Figure()
    fig.add_hline(y=0, line_color=INK, line_dash="dash", line_width=1.2, opacity=0.5)
    for city in CITIES:
        st = CITY_STYLE[city]
        fig.add_trace(go.Scatter(
            x=LEADS, y=bias_c.loc[city], mode="lines+markers", name=city,
            line=dict(color=st["color"], width=2, dash=st["dash"]),
            marker=dict(size=8, symbol=st["symbol"], color=st["color"]),
            hovertemplate=f"<b>{city}</b><br>Lead %{{x}}: %{{y:+.2f}} °C<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=LEADS, y=bias_pooled.values, mode="lines+markers", name="All cities",
        line=dict(color=HI, width=4.5),
        marker=dict(size=11, color=HI, line=dict(color=BG, width=2)),
        hovertemplate="<b>All</b><br>Lead %{x}: %{y:+.2f} °C<extra></extra>"))
    fig.update_xaxes(tickmode="array", tickvals=LEADS, ticktext=[f"{k}d" for k in LEADS])
    theme(fig, "Too warm or too cool? Signed bias by lead",
          "Lead time", "Mean error: forecast − station (°C)")
    hot = bias_c[1].idxmax()
    cap = (f"Above zero = too warm. {hot} carries the biggest day-1 warm bias "
           f"({bias_c.loc[hot, 1]:+.2f} °C). Part of any constant offset comes from the model grid "
           f"point being 3–6 km from each station.")
    return fig, cap

In [75]:
def p_surface():
    m_lead = T.pivot_table(index="month", columns="lead", values="ae", aggfunc="mean"
                           ).reindex(index=range(1, 13), columns=LEADS)
    z = m_lead.values
    fig = go.Figure(go.Surface(
        z=z, x=LEADS, y=MONTHS, colorscale=NEON_SCALE,
        contours={"z": {"show": True, "usecolormap": True, "highlightcolor": HI}},
        showscale=True,
        colorbar=dict(title=dict(text="MAE (°C)", font=dict(size=10, color=MUTED)), thickness=10),
        hovertemplate="Lead %{x}<br>Month %{y}<br>MAE %{z:.2f} °C<extra></extra>"))
    fig.update_layout(scene=dict(
        xaxis=dict(title="Lead", gridcolor=GRID, backgroundcolor="rgba(0,0,0,0)", color=MUTED),
        yaxis=dict(title="Month", gridcolor=GRID, backgroundcolor="rgba(0,0,0,0)", color=MUTED),
        zaxis=dict(title="MAE (°C)", gridcolor=GRID, backgroundcolor="rgba(0,0,0,0)", color=MUTED),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=1.3)),
        bgcolor="rgba(0,0,0,0)"))
    theme(fig, "3D seasonal landscape — error across month × lead", h=580, legend=False)
    hot_m = MONTHS[int(np.nanargmax(z.mean(axis=1)))]
    cap = (f"Height = MAE. The surface is dragged upward both by lead and by the hottest month. "
           f"{hot_m} is the hardest month on average. Drag the plot to rotate.")
    return fig, cap

## Tab 3 — Rain (6)

In [76]:
def p_sankey():
    r1 = R[R["lead"] == 1]
    ow = r1["obs"] >= RAIN_WET_MM; fw = r1["fc"] >= RAIN_WET_MM
    hit = int((ow & fw).sum()); miss = int((ow & ~fw).sum())
    fa = int((~ow & fw).sum()); cn = int((~ow & ~fw).sum())
    total = hit + miss + fa + cn
    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(pad=26, thickness=26,
                  line=dict(color="rgba(0,0,0,0)", width=0),
                  label=[f"Obs dry · {cn:,}", f"Obs wet · {hit+miss:,}",
                         f"Fc dry · {miss+cn:,}", f"Fc wet · {hit+fa:,}"],
                  color=[VIOLET, CYAN, VIOLET, PINK]),
        link=dict(source=[0, 0, 1, 1], target=[2, 3, 2, 3],
                  value=[cn, fa, miss, hit],
                  color=[rgba(VIOLET, 0.30), rgba(PINK, 0.35),
                         rgba(PINK, 0.35), rgba(CYAN, 0.55)])))
    theme(fig, f"Where do {total:,} day-1 rain pairs go?", h=500, legend=False)
    cap = (f"Hits = forecast wet & it rained; misses = forecast dry but it rained; "
           f"false alarms = forecast wet but dry. Day-1 CSI: {100*hit/(hit+miss+fa):.0f}%.")
    return fig, cap

In [77]:
def p_confusion():
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Lead 1 day", "Lead 7 days"),
                        shared_yaxes=True, horizontal_spacing=0.07)
    for i, k in enumerate((1, 7), 1):
        d = R[R["lead"] == k]
        ow = d["obs"] >= RAIN_WET_MM; fw = d["fc"] >= RAIN_WET_MM
        hit = int((ow & fw).sum()); miss = int((ow & ~fw).sum())
        fa = int((~ow & fw).sum()); cn = int((~ow & ~fw).sum())
        z = [[cn, fa], [miss, hit]]
        txt = [[f"{cn:,}<br>{100*cn/max(cn+fa,1):.0f}%", f"{fa:,}<br>{100*fa/max(cn+fa,1):.0f}%"],
               [f"{miss:,}<br>{100*miss/max(miss+hit,1):.0f}%", f"{hit:,}<br>{100*hit/max(miss+hit,1):.0f}%"]]
        kw = dict(z=z, x=["Fc dry", "Fc wet"], y=["Obs dry", "Obs wet"],
                  text=txt, texttemplate="%{text}", colorscale=NEON_SCALE,
                  showscale=(i == 2), xgap=4, ygap=4,
                  textfont=dict(size=12, family="JetBrains Mono", color=INK),
                  hovertemplate="%{y} → %{x}<br>%{z:,} days<extra></extra>")
        if i == 2:
            kw["colorbar"] = dict(title=dict(text="Days", font=dict(size=10, color=MUTED)), thickness=10)
        fig.add_trace(go.Heatmap(**kw), row=1, col=i)
    fig.update_yaxes(autorange="reversed")
    theme(fig, "Rain confusion matrix — fewer hits, more errors", legend=False)
    cap = ("Each row sums to 100%; bottom-right = catch rate. From day 1 to day 7 the share of dry days "
           "wrongly called wet grows, and the share of wet days missed grows too.")
    return fig, cap

In [78]:
def p_rain_breakdown():
    parts = []
    for k in LEADS:
        d = R[R["lead"] == k]
        ow = d["obs"] >= RAIN_WET_MM; fw = d["fc"] >= RAIN_WET_MM
        hit = int((ow & fw).sum()); miss = int((ow & ~fw).sum()); fa = int((~ow & fw).sum())
        tot = hit + miss + fa
        parts.append(dict(lead=k,
                          hit=100*hit/tot if tot else 0,
                          miss=100*miss/tot if tot else 0,
                          fa=100*fa/tot if tot else 0))
    df = pd.DataFrame(parts).set_index("lead")
    fig = go.Figure()
    fig.add_trace(go.Bar(x=LEADS, y=df["hit"], name="Hits",
        marker=dict(color=CYAN, line=dict(color=BG, width=1)),
        text=[f"{v:.0f}%" for v in df["hit"]], textposition="inside",
        textfont=dict(color=BG, size=11, family="JetBrains Mono"),
        hovertemplate="Hits: %{y:.0f}%<extra></extra>"))
    fig.add_trace(go.Bar(x=LEADS, y=df["miss"], name="Misses",
        marker=dict(color=VIOLET, line=dict(color=BG, width=1)),
        text=[f"{v:.0f}%" for v in df["miss"]], textposition="inside",
        textfont=dict(color=INK, size=11, family="JetBrains Mono"),
        hovertemplate="Misses: %{y:.0f}%<extra></extra>"))
    fig.add_trace(go.Bar(x=LEADS, y=df["fa"], name="False alarms",
        marker=dict(color=PINK, line=dict(color=BG, width=1)),
        text=[f"{v:.0f}%" for v in df["fa"]], textposition="inside",
        textfont=dict(color=INK, size=11, family="JetBrains Mono"),
        hovertemplate="False alarms: %{y:.0f}%<extra></extra>"))
    fig.update_layout(barmode="stack")
    fig.update_xaxes(tickmode="array", tickvals=LEADS, ticktext=[f"{k}d" for k in LEADS])
    fig.update_yaxes(range=[0, 100], ticksuffix="%")
    theme(fig, "When it rains or doesn't — how each rain day is called",
          "Lead time", "Share of rain-event days (%)")
    cap = (f"Stacked hit / miss / false-alarm shares across all rain-event days. Hits fall "
           f"{df.loc[1,'hit']:.0f}% → {df.loc[7,'hit']:.0f}%; misses and false alarms take the remainder.")
    return fig, cap

In [79]:
def p_pod_sr():
    df = pod_sr_by_lead(R)
    city_rows = []
    for city in CITIES:
        d = pod_sr_by_lead(R[R["city"] == city])
        for k in LEADS:
            city_rows.append(dict(city=city, lead=k,
                                  pod=d.loc[k, "pod"], sr=d.loc[k, "sr"]))
    cdf = pd.DataFrame(city_rows)

    gx = np.linspace(0.05, 1, 40)
    zz = 1 / (1 / gx[None, :] + 1 / gx[:, None] - 1)
    fig = go.Figure(go.Contour(
        z=zz, x=gx, y=gx, colorscale=[[0, "rgba(20,20,31,0)"], [1, "rgba(60,60,90,0.55)"]],
        opacity=0.5, contours=dict(showlabels=True, start=0.1, end=0.9, size=0.1,
                                   labelfont=dict(size=9, color=MUTED)),
        line=dict(width=0.5, color=GRID),
        showscale=False, hoverinfo="skip", showlegend=False))
    for city in CITIES:
        st = CITY_STYLE[city]
        d = cdf[cdf["city"] == city].sort_values("lead")
        fig.add_trace(go.Scatter(
            x=d["sr"], y=d["pod"], mode="lines+markers", name=city,
            line=dict(color=st["color"], width=2, dash=st["dash"]),
            marker=dict(size=6, symbol=st["symbol"], color=st["color"]),
            text=d["lead"].astype(str),
            hovertemplate=f"<b>{city}</b><br>lead %{{text}}<br>SR %{{x:.2f}}, POD %{{y:.2f}}<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=df["sr"], y=df["pod"], mode="lines+markers+text",
        name="All cities", text=[str(k) for k in LEADS],
        textposition="top right", textfont=dict(color=HI, size=13, family="JetBrains Mono"),
        line=dict(color=HI, width=3.5),
        marker=dict(size=11, color=HI, line=dict(color=BG, width=2)),
        hovertemplate="All<br>lead %{text}<br>SR %{x:.2f}, POD %{y:.2f}<extra></extra>"))
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    theme(fig, "Detection vs precision — the trade-off path",
          "Success ratio (1 − false-alarm ratio)", "Probability of detection")
    cap = (f"Up-right is better. Grey contours = equal CSI. The amber path walks from "
           f"POD {df.loc[1,'pod']:.2f}/SR {df.loc[1,'sr']:.2f} at day 1 to "
           f"{df.loc[7,'pod']:.2f}/{df.loc[7,'sr']:.2f} at day 7.")
    return fig, cap

In [80]:
def p_csi_thresh():
    thrs = [1, 5, 10, 25]
    z, yl = [], []
    for thr in thrs:
        s = pd.DataFrame({
            "lead": R["lead"],
            "h": ((R["obs"] >= thr) & (R["fc"] >= thr)).astype(int),
            "m": ((R["obs"] >= thr) & (R["fc"] < thr)).astype(int),
            "f": ((R["obs"] < thr) & (R["fc"] >= thr)).astype(int),
        }).groupby("lead").sum().reindex(LEADS)
        csi = s["h"] / (s["h"] + s["m"] + s["f"]).replace(0, np.nan)
        z.append(csi.values)
        yl.append(f"≥ {thr} mm")
    z = np.array(z, float)
    fig = go.Figure(go.Heatmap(
        z=z, x=[f"{k}d" for k in LEADS], y=yl, colorscale=NEON_SCALE,
        zmin=0, zmax=1,
        text=np.round(z, 2), texttemplate="%{text}",
        textfont=dict(size=12, family="JetBrains Mono", color=INK),
        xgap=3, ygap=3,
        colorbar=dict(title=dict(text="CSI", font=dict(size=10, color=MUTED)), thickness=10),
        hovertemplate="%{y}<br>%{x}<br>CSI %{z:.2f}<extra></extra>"))
    fig.update_yaxes(autorange="reversed")
    theme(fig, "Rain skill by intensity — heavier rain is harder",
          "Lead time", None, legend=False)
    cap = ("Rows are wet-day thresholds. CSI drops from day 1 to day 7 in every row, and from lighter to "
           "heavier rain at every lead — both axes push the score down.")
    return fig, cap

In [81]:
def p_qq():
    ticks = [0, 1, 3, 10, 30, 100]
    fig = go.Figure()
    pcts = np.arange(1, 100)
    for k, col in ((1, CYAN), (7, PINK)):
        d = R[R["lead"] == k]
        fo = np.log1p(np.percentile(d["obs"], pcts))
        ff = np.log1p(np.percentile(d["fc"], pcts))
        fig.add_trace(go.Scatter(
            x=fo, y=ff, mode="lines", name=f"Lead {k}",
            line=dict(color=col, width=3.2), customdata=pcts,
            hovertemplate=f"<b>Lead {k}</b><br>%{{customdata}}th pct<br>"
                          f"log-obs %{{x:.2f}} · log-fc %{{y:.2f}}<extra></extra>"))
    top = float(np.log1p(max(R["obs"].max(), R["fc"].max())))
    fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines",
                             name="Perfect", line=dict(color=INK, dash="dot", width=1.2),
                             hoverinfo="skip"))
    fig.update_xaxes(tickmode="array", tickvals=[float(np.log1p(v)) for v in ticks],
                     ticktext=[str(v) for v in ticks])
    fig.update_yaxes(tickmode="array", tickvals=[float(np.log1p(v)) for v in ticks],
                     ticktext=[str(v) for v in ticks])
    theme(fig, "Rain QQ — does the forecast under-produce heavy rain?",
          "Station rain percentile (mm, log)", "Forecast rain percentile (mm, log)")
    q7o = float(np.percentile(R[R["lead"] == 7]["obs"], 95))
    q7f = float(np.percentile(R[R["lead"] == 7]["fc"], 95))
    cap = (f"Below the dashed line means the forecast quantile is lower than the station's — "
           f"it under-produces heavy rain. At day 7 the 95th percentile of forecast rain is "
           f"{q7f:.0f} mm vs {q7o:.0f} mm observed.")
    return fig, cap

## Tab 4 — Monsoon (6)

In [82]:
def p_calendar():
    T2 = T.copy()
    T2["ym"] = T2["date"].dt.to_period("M")
    months = sorted(T2["ym"].unique())
    z = np.full((3, len(months)), np.nan)
    for i, city in enumerate(CITIES):
        g = T2[(T2["city"] == city) & (T2["lead"] == 7)].groupby("ym")["ae"].mean()
        for j, m in enumerate(months):
            if m in g.index:
                z[i, j] = g[m]
    fig = go.Figure(go.Heatmap(
        z=z, x=[m.strftime("%b %y") for m in months], y=CITIES,
        colorscale=NEON_SCALE, xgap=1, ygap=2,
        colorbar=dict(title=dict(text="MAE (°C)", font=dict(size=10, color=MUTED)), thickness=10),
        hovertemplate="%{y}<br>%{x}<br>Day-7 MAE %{z:.2f} °C<extra></extra>"))
    fig.update_yaxes(autorange="reversed", tickfont=dict(color=INK))
    fig.update_xaxes(tickangle=-45, dtick=3)
    theme(fig, "Day-7 temperature error, city by month", h=400, legend=False)
    cap = ("Each cell is the mean day-7 MAE for one city in one month. The brightest streak is the "
           "pre-monsoon build-up (April–June); monsoon months (Jun–Sep) sit mid-heat.")
    return fig, cap

In [83]:
def p_polar():
    fig = go.Figure()
    for city in CITIES:
        g = T[(T["city"] == city) & (T["lead"] == 7)].groupby("month")["ae"].mean().reindex(range(1, 13))
        if g.isna().all():
            continue
        st = CITY_STYLE[city]
        vals = g.fillna(0).tolist()
        vals.append(vals[0])
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=MONTHS + [MONTHS[0]], name=city, fill="toself",
            fillcolor=rgba(st["color"], 0.12),
            line=dict(color=st["color"], width=2.4, dash=st["dash"]),
            marker=dict(size=6, symbol=st["symbol"], color=st["color"]),
            hovertemplate=f"<b>{city}</b><br>%{{theta}}<br>Day-7 MAE %{{r:.2f}} °C<extra></extra>"))
    fig.update_layout(polar=dict(
        bgcolor="rgba(0,0,0,0)",
        radialaxis=dict(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=MUTED, size=10)),
        angularaxis=dict(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=INK, size=11))))
    theme(fig, "The seasonal clock — day-7 error around the year", h=520)
    cap = ("Angle = month (Jan at top, clockwise); radius = day-7 MAE. Each shape shows which months "
           "are hardest per city. Monsoon months (Jun–Sep) usually push a lobe outward.")
    return fig, cap

In [84]:
def p_box():
    d7 = T[T["lead"] == 7]
    fig = go.Figure()
    for m in range(1, 13):
        vals = d7[d7["month"] == m]["ae"].dropna()
        if vals.empty:
            continue
        col = PINK if m in MONSOON else CYAN
        fig.add_trace(go.Box(
            y=vals, name=MONTHS[m - 1], marker_color=col, line_color=col,
            fillcolor=rgba(col, 0.22), boxpoints=False, showlegend=False,
            hovertemplate=f"<b>{MONTHS[m-1]}</b><br>MAE %{{y:.2f}} °C<extra></extra>"))
    theme(fig, "Month-by-month day-7 error — monsoon months (pink) are wider",
          None, "Absolute error (°C)", legend=False)
    cap = ("Each box is one month's absolute error at day 7, pooled across cities. Pink = monsoon "
           "(Jun–Sep). Both the median and interquartile spread rise through the monsoon months.")
    return fig, cap

In [85]:
def p_bootstrap():
    rng = np.random.default_rng(7)
    rows = []
    for k in LEADS:
        d = T[T["lead"] == k]
        mon = d.loc[d["month"].isin(MONSOON), "ae"].to_numpy()
        oth = d.loc[~d["month"].isin(MONSOON), "ae"].to_numpy()
        if len(mon) < 5 or len(oth) < 5:
            rows.append((k, np.nan, np.nan, np.nan)); continue
        gap = mon.mean() - oth.mean()
        diffs = np.empty(400)
        for i in range(400):
            a = rng.choice(mon, len(mon), replace=True).mean()
            b = rng.choice(oth, len(oth), replace=True).mean()
            diffs[i] = a - b
        rows.append((k, gap, np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)))
    g = pd.DataFrame(rows, columns=["lead", "gap", "lo", "hi"])
    valid = g.dropna(subset=["gap", "lo", "hi"])
    sig = (valid["lo"] > 0) | (valid["hi"] < 0)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=valid["lead"], y=valid["gap"], mode="markers",
        marker=dict(size=15, color=[PINK if s else CYAN for s in sig],
                    line=dict(color=BG, width=2)),
        error_y=dict(type="data", symmetric=False,
                     array=(valid["hi"] - valid["gap"]).tolist(),
                     arrayminus=(valid["gap"] - valid["lo"]).tolist(),
                     color=MUTED, thickness=1.6, width=7),
        hovertemplate="Lead %{x}<br>Gap %{y:+.2f} °C<extra></extra>"))
    fig.add_hline(y=0, line_color=INK, line_dash="dash", line_width=1.2)
    fig.update_xaxes(tickmode="array", tickvals=LEADS, ticktext=[f"{k}d" for k in LEADS])
    theme(fig, "How much harder is the monsoon? Monsoon − other MAE with 95% CI",
          "Lead time", "MAE gap (°C)")
    n_sig = int(sig.sum())
    cap = (f"Points = monsoon minus other-month MAE; bars = 400-resample bootstrap 95% intervals. "
           f"Pink = interval excludes zero ({n_sig} of {len(valid)} valid leads). "
           f"Days treated as independent, so intervals are optimistic.")
    return fig, cap

In [86]:
def p_monsoon_within():
    d7 = T[T["lead"] == 7]
    rows = []
    for city in CITIES:
        for season in SEASONS:
            sub = d7[(d7["city"] == city) & (d7["season"] == season)]
            pct = 100 * (sub["ae"] <= TEMP_THR).mean() if len(sub) else np.nan
            rows.append(dict(city=city, season=season, within=pct, n=len(sub)))
    df = pd.DataFrame(rows)
    fig = go.Figure()
    for season, col in [(SEASONS[0], PINK), (SEASONS[1], CYAN)]:
        sub = df[df["season"] == season]
        fig.add_trace(go.Bar(
            x=sub["city"], y=sub["within"], name=season,
            marker=dict(color=col, line=dict(color=BG, width=1)),
            text=[f"{v:.0f}%" for v in sub["within"]],
            textposition="outside",
            textfont=dict(color=col, size=11, family="JetBrains Mono"),
            hovertemplate=f"<b>%{{x}}</b><br>{season}: %{{y:.0f}}% within {TEMP_THR:.0f} °C<extra></extra>"))
    fig.update_layout(barmode="group", bargap=0.3)
    fig.update_yaxes(range=[0, 110], ticksuffix="%")
    theme(fig, f"Day 7: share of days within {TEMP_THR:.0f} °C, monsoon vs other",
          None, f"% within {TEMP_THR:.0f} °C")
    cap = (f"At day 7, monsoon days are less often within {TEMP_THR:.0f} °C than other-month days "
           f"in every city. The pooled monsoon pass rate is "
           f"{df[df['season']==SEASONS[0]]['within'].mean():.0f}% vs "
           f"{df[df['season']==SEASONS[1]]['within'].mean():.0f}% outside.")
    return fig, cap

In [87]:
def p_monsoon_city():
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Lead 1 day", "Lead 7 days"),
                        shared_yaxes=True, horizontal_spacing=0.07)
    for j, k in enumerate((1, 7), 1):
        d = T[T["lead"] == k]
        for i, city in enumerate(CITIES):
            a = d[(d["city"] == city) & (~d["month"].isin(MONSOON))]["ae"].mean()
            b = d[(d["city"] == city) & (d["month"].isin(MONSOON))]["ae"].mean()
            if pd.isna(a) or pd.isna(b):
                continue
            fig.add_trace(go.Scatter(
                x=[a, b], y=[city, city], mode="lines",
                line=dict(color=GRID, width=6), showlegend=False, hoverinfo="skip"),
                row=1, col=j)
            for nm, v, col, sym, pos in (("Other months", a, CYAN, "diamond", "bottom center"),
                                          ("Monsoon (Jun-Sep)", b, PINK, "circle", "top center")):
                fig.add_trace(go.Scatter(
                    x=[v], y=[city], mode="markers+text",
                    name=nm, legendgroup=nm, showlegend=(i == 0 and j == 1),
                    text=[f"{v:.2f}"],
                    textposition=pos,
                    textfont=dict(size=10, color=INK, family="JetBrains Mono"),
                    marker=dict(size=14, color=col, symbol=sym,
                                line=dict(color=BG, width=1.5)),
                    hovertemplate=f"<b>{city}</b><br>{nm}<br>MAE %{{x:.2f}} °C<extra></extra>"),
                    row=1, col=j)
    fig.update_xaxes(title_text="MAE (°C)", rangemode="tozero")
    fig.update_yaxes(autorange="reversed", tickfont=dict(color=INK))
    theme(fig, "Monsoon gap by city — do all three agree?")
    cap = ("Each row joins a city's other-month error (cyan diamond) to its monsoon error (pink dot). "
           "An amber dot to the right of the diamond means the monsoon is harder there.")
    return fig, cap

## Tab 5 — Cities (5)

In [88]:
def p_radar():
    t7 = T[T["lead"] == 7].groupby("city").agg(
        mae=("ae", "mean"), bias=("err", "mean"), sd=("err", "std")).reindex(CITIES)
    r7 = R[R["lead"] == 7].copy()
    r_mae = r7.groupby("city")["ae"].mean().reindex(CITIES)
    r7["correct"] = ((r7["fc"] >= RAIN_WET_MM) == (r7["obs"] >= RAIN_WET_MM)).astype(int)
    r_err = (1 - r7.groupby("city")["correct"].mean()).reindex(CITIES)
    raw = pd.DataFrame({
        "Temp MAE": t7["mae"], "Temp |bias|": t7["bias"].abs(),
        "Temp SD": t7["sd"], "Rain MAE": r_mae, "Rain error rate": r_err,
    }).reindex(CITIES)
    nm = (raw / raw.max()).fillna(0)
    cats = list(raw.columns)
    fig = go.Figure()
    for city in CITIES:
        st = CITY_STYLE[city]
        v = nm.loc[city].tolist()
        fig.add_trace(go.Scatterpolar(
            r=v + v[:1], theta=cats + cats[:1], name=city, fill="toself",
            fillcolor=rgba(st["color"], 0.14),
            line=dict(color=st["color"], width=2.4, dash=st["dash"]),
            marker=dict(size=7, symbol=st["symbol"], color=st["color"]),
            hovertemplate=f"<b>{city}</b><br>%{{theta}}: %{{r:.2f}} of worst<extra></extra>"))
    fig.update_layout(polar=dict(
        bgcolor="rgba(0,0,0,0)",
        radialaxis=dict(range=[0, 1.05], gridcolor=GRID, linecolor=GRID,
                        tickfont=dict(color=MUTED, size=10)),
        angularaxis=dict(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=INK, size=11))))
    theme(fig, "Day-7 city fingerprint — bigger shape = harder to forecast", h=520)
    worst = nm.idxmax()
    cap = (f"Each axis is normalised so the worst city scores 1. "
           f"{worst.value_counts().index[0]} tops {int(worst.value_counts().iloc[0])} of 5 axes. "
           f"The scaling exaggerates small gaps — read with the heatmaps.")
    return fig, cap

In [89]:
def p_city_heatmap():
    mae_c = T.groupby(["city", "lead"])["ae"].mean().unstack().reindex(CITIES)
    fig = go.Figure(go.Heatmap(
        z=mae_c.values, x=[f"{k}d" for k in LEADS], y=CITIES,
        colorscale=NEON_SCALE,
        text=np.round(mae_c.values, 2), texttemplate="%{text}",
        textfont=dict(size=12, family="JetBrains Mono", color=INK),
        xgap=3, ygap=3,
        colorbar=dict(title=dict(text="MAE (°C)", font=dict(size=10, color=MUTED)), thickness=10),
        hovertemplate="%{y}<br>%{x}: %{z:.2f} °C<extra></extra>"))
    fig.update_yaxes(autorange="reversed", tickfont=dict(color=INK))
    theme(fig, "Temperature MAE by city × lead", "Lead time", None, legend=False)
    l7 = mae_c[7]
    cap = (f"{l7.idxmax()} has the largest day-7 error ({l7.max():.2f} °C); {l7.idxmin()} the smallest "
           f"({l7.min():.2f} °C). Across all leads {mae_c.mean(axis=1).idxmax()} averages "
           f"{mae_c.mean(axis=1).max():.2f} °C.")
    return fig, cap

In [90]:
def p_distance_bias():
    b = T[T["lead"] == 1].groupby("city")["err"].agg(["mean", "std", "size"]).reindex(CITIES)
    ci = 1.96 * b["std"] / np.sqrt(b["size"])
    labels = [f"{c}<br><span style='font-size:10px;color:{MUTED}'>grid {GRID_KM[c]:.1f} km away</span>" for c in CITIES]
    fig = go.Figure(go.Bar(
        x=labels, y=b["mean"].values,
        marker=dict(color=[CITY_STYLE[c]["color"] for c in CITIES],
                    line=dict(color=BG, width=1)),
        error_y=dict(type="data", array=ci.values, color=HI, thickness=2.5, width=7),
        text=[f"{v:+.2f}" for v in b["mean"].values], textposition="outside",
        textfont=dict(color=INK, size=12, family="JetBrains Mono"),
        hovertemplate="%{x}<br>Mean error %{y:+.2f} °C<extra></extra>"))
    fig.add_hline(y=0, line_color=INK, line_width=1.2, line_dash="dash")
    theme(fig, "Even at day 1 the forecast sits off the station — the error floor",
          None, "Mean day-1 error (°C)", legend=False)
    cap = (f"Bars are the mean day-1 error with 95% intervals. This is the error before any skill is lost. "
           f"The offset is partly the grid-point gap ({min(GRID_KM.values()):.1f}–{max(GRID_KM.values()):.1f} km) "
           f"and partly altitude or coast effects the tables cannot separate.")
    return fig, cap

In [91]:
def p_city_slope():
    mae_c = T.groupby(["city", "lead"])["ae"].mean().unstack().reindex(CITIES)
    d1, d7 = mae_c[1], mae_c[7]
    order = d7.sort_values().index.tolist()
    fig = go.Figure()
    for c in order:
        fig.add_trace(go.Scatter(x=[d1[c], d7[c]], y=[c, c], mode="lines",
                                 line=dict(color=GRID, width=6),
                                 showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(
        x=[d1[c] for c in order], y=order, mode="markers+text", name="Lead 1",
        text=[f"{d1[c]:.2f}" for c in order], textposition="middle left",
        textfont=dict(color=CYAN, size=11, family="JetBrains Mono"),
        marker=dict(size=15, color=CYAN, symbol="diamond", line=dict(color=BG, width=1.5))))
    fig.add_trace(go.Scatter(
        x=[d7[c] for c in order], y=order, mode="markers+text", name="Lead 7",
        text=[f"{d7[c]:.2f}" for c in order], textposition="middle right",
        textfont=dict(color=PINK, size=11, family="JetBrains Mono"),
        marker=dict(size=15, color=PINK, line=dict(color=BG, width=1.5))))
    fig.update_xaxes(range=[0, float(d7.max()) * 1.28])
    fig.update_yaxes(tickfont=dict(color=INK))
    theme(fig, "Day 1 → day 7 — each city's accuracy lost in a week",
          "MAE (°C)", None)
    lose = (d7 - d1)
    cap = (f"Cyan diamond = day 1, pink dot = day 7. {lose.idxmax()} loses the most "
           f"({d1[lose.idxmax()]:.2f} → {d7[lose.idxmax()]:.2f} °C, a {lose.max():.2f} °C hit); "
           f"{lose.idxmin()} loses the least ({lose.min():.2f} °C).")
    return fig, cap

In [92]:
def p_bubble():
    t7 = T[T["lead"] == 7].groupby("city").agg(mae=("ae", "mean"), n=("ae", "size")).reindex(CITIES)
    r7 = R[R["lead"] == 7].groupby("city").agg(mae=("ae", "mean"), n=("ae", "size")).reindex(CITIES)
    fig = go.Figure()
    for city in CITIES:
        st = CITY_STYLE[city]
        fig.add_trace(go.Scatter(
            x=[t7.loc[city, "mae"]], y=[r7.loc[city, "mae"]],
            mode="markers+text", name=city,
            text=[city], textposition="top center",
            textfont=dict(color=st["color"], size=12, family="Space Grotesk"),
            marker=dict(size=44, color=st["color"], opacity=0.6,
                        line=dict(color=st["color"], width=2)),
            hovertemplate=f"<b>{city}</b><br>Day-7 Temp MAE %{{x:.2f}} °C<br>Day-7 Rain MAE %{{y:.1f}} mm<extra></extra>"))
    theme(fig, "Day-7 trade-off — who pays for temperature, who for rain?",
          "Temperature MAE at day 7 (°C)", "Rain MAE at day 7 (mm/day)")
    cap = ("Upper-right = harder on both axes. Bubble size is fixed; axes are day-7 MAE from each city's own "
           "matched days. Different cities lead each axis, so 'worst city' depends on which weather you care about.")
    return fig, cap

## Tab 6 — Quality + Before/After (6)

In [93]:
def p_before_after():
    cal_t = len(pd.date_range(WIN_START, WIN_END))
    cal_r = len(pd.date_range(RAIN_START, WIN_END))
    clean_t = temp_wide.groupby("city").size().reindex(CITIES).fillna(0)
    clean_r = rain_wide.groupby("city").size().reindex(CITIES).fillna(0)
    raw_t = pd.Series(0, index=CITIES, dtype=float)
    raw_r = pd.Series(0, index=CITIES, dtype=float)
    if INTERIM_OK:
        im = INTERIM
        rt = im[(im["element"] == "TMAX") & im["date"].between(WIN_START, WIN_END)]
        rr = im[(im["element"] == "PRCP") & im["date"].between(RAIN_START, WIN_END)]
        raw_t = rt.groupby("city").size().reindex(CITIES).fillna(0)
        raw_r = rr.groupby("city").size().reindex(CITIES).fillna(0)

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f"Temperature (TMAX) — {cal_t:,} calendar days/city",
                                        f"Rain (PRCP) — {cal_r:,} calendar days/city"),
                        horizontal_spacing=0.1)
    for j, (cal, raw_s, clean_s) in enumerate(
            ((cal_t, raw_t, clean_t), (cal_r, raw_r, clean_r)), 1):
        fig.add_trace(go.Bar(
            name="Calendar days", x=CITIES, y=[cal] * 3,
            marker=dict(color=GRID, line=dict(color=EDGE, width=1)),
            legendgroup="cal", showlegend=(j == 1),
            hovertemplate="<b>%{x}</b><br>Calendar days: %{y:,}<extra></extra>"),
            row=1, col=j)
        if INTERIM_OK and raw_s.sum() > 0:
            fig.add_trace(go.Bar(
                name="Raw station rows", x=CITIES, y=raw_s.values,
                marker=dict(color=MUTED, line=dict(color=BG, width=1)),
                legendgroup="raw", showlegend=(j == 1),
                hovertemplate="<b>%{x}</b><br>Raw: %{y:,}<extra></extra>"),
                row=1, col=j)
        fig.add_trace(go.Bar(
            name="Clean matched rows", x=CITIES, y=clean_s.values,
            marker=dict(color=HI, line=dict(color=BG, width=1)),
            legendgroup="clean", showlegend=(j == 1),
            hovertemplate="<b>%{x}</b><br>Clean: %{y:,}<extra></extra>"),
            row=1, col=j)
    fig.update_layout(barmode="group", bargap=0.22)
    fig.update_yaxes(title_text="Rows", row=1, col=1)
    theme(fig, "Before vs after cleaning — how much raw data survives to analysis")
    surv_t = 100 * clean_t.sum() / max(3 * cal_t, 1)
    surv_r = 100 * clean_r.sum() / max(3 * cal_r, 1)
    cap = (f"Grey = calendar days in window; muted = raw station rows before any filter; amber = matched days "
           f"that reach the final analysis. Temperature retains {surv_t:.0f}% of calendar days, rain retains "
           f"{surv_r:.0f}%. Most of the loss is missing station values, not quality flags.")
    return fig, cap

In [94]:
def p_coverage():
    T2 = T.copy()
    T2["ym"] = T2["date"].dt.to_period("M")
    months = pd.period_range(WIN_START.to_period("M"), WIN_END.to_period("M"), freq="M")
    z = np.zeros((3, len(months)))
    for i, city in enumerate(CITIES):
        cnt = T2[T2["city"] == city].groupby("ym").size()
        for j, m in enumerate(months):
            nd = (min(m.end_time.normalize(), WIN_END) - max(m.start_time, WIN_START)).days + 1
            z[i, j] = 100 * cnt.get(m, 0) / nd
    fig = go.Figure(go.Heatmap(
        z=z, x=[m.strftime("%b %y") for m in months], y=CITIES,
        colorscale=NEON_SCALE, zmin=0, zmax=100, xgap=1, ygap=2,
        colorbar=dict(title=dict(text="% days", font=dict(size=10, color=MUTED)), thickness=10),
        hovertemplate="%{y}<br>%{x}<br>Coverage %{z:.0f}%<extra></extra>"))
    fig.update_yaxes(autorange="reversed", tickfont=dict(color=INK))
    fig.update_xaxes(tickangle=-45, dtick=3)
    theme(fig, "Coverage calendar — which months have matched days?", h=400, legend=False)
    cap = (f"Each cell is the share of that month's days with a station value and a forecast. "
           f"Mean coverage is {z.mean():.0f}%; thinnest city-month is {z.min():.0f}%. "
           f"Coverage is not uniform, so per-month sample sizes vary.")
    return fig, cap

In [95]:
def p_waterfall():
    if CLEAN_LOG is None or not {"rows_before", "rows_after", "step"}.issubset(CLEAN_LOG.columns):
        fig = go.Figure()
        fig.add_annotation(text="cleaning_log_*.csv not found next to this script",
                           x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           font=dict(size=14, color=MUTED))
        theme(fig, "Cleaning log missing", legend=False)
        return fig, "Place cleaning_log_*.csv from clean/ next to this script."
    lg = CLEAN_LOG.dropna(subset=["rows_before", "rows_after"]).copy()
    lg = lg[lg["rows_after"] < lg["rows_before"]]
    if lg.empty:
        fig = go.Figure()
        fig.add_annotation(text="No cleaning step removed rows",
                           x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           font=dict(size=14, color=MUTED))
        theme(fig, "No row-dropping steps", legend=False)
        return fig, "No step in the cleaning log removed any rows."
    lg["removed"] = lg["rows_before"] - lg["rows_after"]
    lg = lg.sort_values("removed")
    labels = [wrap(str(s), 30) for s in lg["step"]]
    cols = [PINK if i == len(lg) - 1 else CYAN for i in range(len(lg))]
    fig = go.Figure(go.Bar(
        y=labels, x=lg["removed"], orientation="h",
        marker=dict(color=cols, line=dict(color=BG, width=1)),
        text=[f"{int(r):,}" for r in lg["removed"]],
        textposition="outside",
        textfont=dict(color=INK, size=10, family="JetBrains Mono"),
        hovertemplate="%{y}<br>removed %{x:,} rows<extra></extra>"))
    fig.update_layout(margin=dict(l=280))
    theme(fig, f"What cleaning removed ({int(lg['removed'].sum()):,} rows across {len(lg)} steps)",
          "Rows removed", None, legend=False, h=max(380, 34 * len(lg)))
    top = lg.iloc[-1]
    cap = (f"Only steps that deleted rows are shown; the largest is '{top['step']}' "
           f"({int(top['removed']):,} rows). Value-fixing steps are in the log but removed nothing.")
    return fig, cap

In [96]:
def p_outliers():
    d = T.copy()
    d["anom"] = d.groupby(["city", "month"])["obs"].transform(lambda s: s - s.median())
    flag = np.zeros(len(d), bool)
    for city in CITIES:
        m = (d["city"] == city).to_numpy()
        x = d.loc[m, "anom"].to_numpy()
        med = np.median(x)
        mad = np.median(np.abs(x - med))
        if mad > 0:
            flag[m] = np.abs(0.6745 * (x - med) / mad) > 3.5
    d["flag"] = flag
    ok, bad = d[~d["flag"]], d[d["flag"]]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=ok["date"], y=ok["anom"], mode="markers", name="Within range",
        marker=dict(size=3.5, color=MUTED, opacity=0.5),
        customdata=ok["city"],
        hovertemplate="%{customdata}<br>%{x|%d %b %Y}<br>%{y:+.1f} °C<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=bad["date"], y=bad["anom"], mode="markers", name="Flagged (>3.5 MAD)",
        marker=dict(size=11, color=PINK, line=dict(color=BG, width=1.5)),
        customdata=bad["city"],
        hovertemplate="<b>%{customdata}</b><br>%{x|%d %b %Y}<br>%{y:+.1f} °C<extra></extra>"))
    fig.add_hline(y=0, line_color=INK, line_dash="dot", line_width=1, opacity=0.4)
    fig.update_xaxes(range=[str(WIN_START.date()), str(WIN_END.date())], type="date")
    theme(fig, f"Station TMAX anomalies — {len(bad)} flagged of {len(d):,}",
          "Valid date", "Anomaly from city-month median (°C)")
    cap = (f"Each dot is station TMAX minus the median of the same city and month. Pink = modified z-score "
           f">3.5. The forecast is deliberately not used to judge outliers — that would be circular.")
    return fig, cap

In [97]:
def p_rain_wet_share():
    R2 = R.copy()
    R2["wet"] = (R2["obs"] >= RAIN_WET_MM).astype(int)
    d1 = R2[R2["lead"] == 1]
    fig = go.Figure()
    for city in CITIES:
        d = d1[d1["city"] == city]
        fig.add_trace(go.Bar(
            x=[city], y=[100 * d["wet"].mean()], name=city,
            marker=dict(color=CITY_STYLE[city]["color"], line=dict(color=BG, width=1)),
            text=[f"{100 * d['wet'].mean():.0f}%"], textposition="outside",
            textfont=dict(color=INK, size=13, family="JetBrains Mono"),
            customdata=[len(d)],
            hovertemplate=f"<b>{city}</b><br>%{{y:.0f}}% wet days<br>n = %{{customdata}}<extra></extra>",
            showlegend=False))
    fig.update_yaxes(range=[0, 100], ticksuffix="%")
    theme(fig, "Rain sample leans wet — share of recorded days with ≥ 1 mm",
          None, "% recorded wet days", legend=False)
    cap = ("After dropping days with no station value, what fraction of surviving days are wet? "
           "Dry days are the ones stations most often leave blank, so the recorded-only sample "
           "over-represents wet days — the truth is probably lower, not higher.")
    return fig, cap

In [98]:
def p_sample_size():
    rows = []
    for city in CITIES:
        for k in LEADS:
            n = len(T[(T["city"] == city) & (T["lead"] == k)])
            rows.append(dict(city=city, lead=k, n=n))
    df = pd.DataFrame(rows)
    fig = go.Figure()
    for city in CITIES:
        st = CITY_STYLE[city]
        d = df[df["city"] == city]
        fig.add_trace(go.Bar(
            x=d["lead"], y=d["n"], name=city,
            marker=dict(color=st["color"], line=dict(color=BG, width=1)),
            hovertemplate=f"<b>{city}</b><br>lead %{{x}}: %{{y}} days<extra></extra>"))
    fig.update_layout(barmode="group", bargap=0.25)
    fig.update_xaxes(tickmode="array", tickvals=LEADS, ticktext=[f"{k}d" for k in LEADS])
    theme(fig, "Sample size per lead — every result rests on these N",
          "Lead time", "Matched days")
    total = int(df["n"].sum())
    cap = (f"Matched days per city and lead. The same days are used across all 7 leads (a day is kept only if "
           f"every lead has a forecast), so the bars are near-flat. Total across cities and leads: {total:,}.")
    return fig, cap

## Safe Wrapper

In [99]:
def safe(fn, name):
    try:
        return fn()
    except Exception as e:
        print(f"  ! plot '{name}' failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        fig = go.Figure()
        fig.add_annotation(text=f"Could not render: {name}", x=0.5, y=0.55,
                           xref="paper", yref="paper", showarrow=False,
                           font=dict(size=14, color=MUTED))
        fig.add_annotation(text=f"{type(e).__name__}: {str(e)[:180]}", x=0.5, y=0.45,
                           xref="paper", yref="paper", showarrow=False,
                           font=dict(size=10, color=MUTED, family="JetBrains Mono"))
        theme(fig, name, legend=False)
        return fig, f"Render failed — see console for details: {type(e).__name__}"

## Tabs

In [100]:
TAB_SPEC = [
    ("Pulse", "The 10-second answer", HI, [
        ("Trust horizon per city", p_trust_bars),
        ("Cone of uncertainty", p_cone),
        ("Scorecard", p_scorecard),
        ("Seasonal pulse", p_rolling),
        ("Where the data lives", p_treemap),
    ]),
    ("Temperature", "How fast does the error grow?", CYAN, [
        ("Error ridge", p_ridge),
        ("Forecast vs station", p_hexbin),
        ("Within tolerance", p_within_tol),
        ("Per-city slope", p_slope),
        ("Bias by lead", p_bias),
        ("3D seasonal landscape", p_surface),
    ]),
    ("Rain", "When it matters, does it know?", PINK, [
        ("Where the days go (Sankey)", p_sankey),
        ("Confusion matrix", p_confusion),
        ("Hit / miss / false-alarm split", p_rain_breakdown),
        ("Detection vs precision", p_pod_sr),
        ("Skill by intensity", p_csi_thresh),
        ("Quantile alignment (QQ)", p_qq),
    ]),
    ("Monsoon", "Is June–September harder?", LIME, [
        ("Calendar of error", p_calendar),
        ("Seasonal clock", p_polar),
        ("Month spread", p_box),
        ("Confidence gap", p_bootstrap),
        ("Within-tol% monsoon vs other", p_monsoon_within),
        ("Monsoon gap per city", p_monsoon_city),
    ]),
    ("Cities", "Who suffers most?", VIOLET, [
        ("City fingerprint (radar)", p_radar),
        ("Temperature by city × lead", p_city_heatmap),
        ("Day-1 error floor", p_distance_bias),
        ("Day 1 → 7 loss", p_city_slope),
        ("Temp vs rain (bubble)", p_bubble),
    ]),
    ("Data quality", "What did cleaning cost us?", ROSE, [
        ("Before vs after cleaning", p_before_after),
        ("Coverage calendar", p_coverage),
        ("Cleaning waterfall", p_waterfall),
        ("Outlier scan", p_outliers),
        ("Rain sample leans wet", p_rain_wet_share),
        ("Sample size per lead", p_sample_size),
    ]),
]

## Css / Js / Template

In [101]:
CSS = """
* { box-sizing: border-box; }
html { background: __BG__; }
body {
  margin: 0;
  font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
  color: __INK__;
  background:
    radial-gradient(1000px 500px at 10% -10%, rgba(34, 211, 238, 0.13), transparent 55%),
    radial-gradient(900px 500px at 95% 5%, rgba(236, 72, 153, 0.10), transparent 55%),
    radial-gradient(800px 600px at 50% 110%, rgba(163, 230, 53, 0.06), transparent 55%),
    __BG__;
  min-height: 100vh;
}
body::before {
  content: ''; position: fixed; inset: 0;
  background-image: radial-gradient(circle, rgba(255,255,255,0.035) 1px, transparent 1px);
  background-size: 22px 22px; pointer-events: none; z-index: 0;
}
.wrap { position: relative; z-index: 1; max-width: 1440px; margin: 0 auto; padding: 40px 32px 60px; }
.eyebrow {
  display: inline-block; padding: 6px 14px;
  background: rgba(34, 211, 238, 0.08);
  border: 1px solid rgba(34, 211, 238, 0.3);
  border-radius: 999px; color: __CYAN__;
  font-size: 0.72rem; font-weight: 700;
  letter-spacing: 0.16em; text-transform: uppercase;
  margin-bottom: 20px; font-family: 'JetBrains Mono', monospace;
}
h1 {
  font-family: 'Space Grotesk', 'Inter', sans-serif;
  font-size: clamp(2rem, 4.5vw, 3.4rem);
  line-height: 1.05; margin: 0 0 16px; font-weight: 700;
  letter-spacing: -0.025em; max-width: 1100px;
}
h1 .grad {
  background: linear-gradient(120deg, __CYAN__ 0%, __PINK__ 45%, __HI__ 100%);
  -webkit-background-clip: text; background-clip: text;
  color: transparent; -webkit-text-fill-color: transparent;
}
.sub { color: __MUTED__; font-size: 1rem; line-height: 1.6; margin: 0 0 28px; max-width: 900px; }
.sub b { color: __INK__; font-weight: 600; }
.tabs {
  position: sticky; top: 0; z-index: 20; display: flex; gap: 8px;
  flex-wrap: wrap; padding: 16px 0; margin-bottom: 18px;
  background: linear-gradient(180deg, __BG__ 60%, rgba(5,5,11,0));
  backdrop-filter: blur(16px); -webkit-backdrop-filter: blur(16px);
}
.tabbtn {
  cursor: pointer; font-family: inherit; font-size: 0.84rem; font-weight: 600;
  color: __MUTED__; background: rgba(14,14,26,0.7);
  border: 1px solid __EDGE__; border-radius: 999px;
  padding: 10px 18px; transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1);
  display: flex; align-items: center; gap: 8px; white-space: nowrap;
}
.tabbtn:hover { color: __INK__; border-color: rgba(255,255,255,0.2); transform: translateY(-1px); }
.tabbtn.on {
  background: var(--accent, __HI__); color: __BG__; border-color: var(--accent, __HI__);
  box-shadow: 0 6px 24px var(--accent-glow, rgba(255,176,32,0.4));
}
.tabbtn .num { font-family: 'JetBrains Mono', monospace; font-size: 0.7rem; opacity: 0.75; font-weight: 700; }
.hero { display: grid; grid-template-columns: 1.4fr 1fr 1fr; gap: 16px; margin-bottom: 24px; }
.kpi {
  position: relative;
  background: linear-gradient(160deg, rgba(20,20,34,0.9) 0%, rgba(14,14,26,0.9) 100%);
  border: 1px solid __EDGE__; border-radius: 16px; padding: 22px 26px;
  overflow: hidden; box-shadow: 0 12px 40px rgba(0,0,0,0.4);
  transition: all 0.3s;
}
.kpi:hover { border-color: rgba(255,255,255,0.15); transform: translateY(-2px); }
.kpi::before {
  content: ''; position: absolute; top: 0; left: 0; right: 0; height: 2px;
  background: linear-gradient(90deg, transparent, var(--kpi-accent, __HI__), transparent);
}
.kpi .label {
  color: __MUTED__; font-size: 0.7rem; letter-spacing: 0.16em;
  text-transform: uppercase; font-weight: 700; font-family: 'JetBrains Mono', monospace;
}
.kpi .value {
  font-family: 'JetBrains Mono', monospace; font-size: 3rem; font-weight: 700;
  line-height: 1.1; margin-top: 8px; color: var(--kpi-accent, __HI__);
  letter-spacing: -0.03em;
}
.kpi.big .value { font-size: 4rem; }
.kpi .value .unit { font-size: 1rem; font-weight: 500; color: __MUTED__; margin-left: 8px; letter-spacing: 0; }
.kpi .note { color: __SOFT__; font-size: 0.82rem; line-height: 1.5; margin-top: 10px; }
.tabhead { margin: 8px 0 18px; padding: 0 2px; }
.tabhead h2 {
  font-family: 'Space Grotesk', 'Inter', sans-serif;
  font-size: 1.6rem; margin: 0; font-weight: 700;
  letter-spacing: -0.02em; color: var(--accent, __INK__);
}
.tabhead p { margin: 6px 0 0; color: __MUTED__; font-size: 0.94rem; }
.grid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 16px; }
.card {
  background: linear-gradient(160deg, rgba(18,18,30,0.85) 0%, rgba(10,10,20,0.85) 100%);
  border: 1px solid __EDGE__; border-radius: 16px;
  padding: 18px 20px 16px; box-shadow: 0 12px 40px rgba(0,0,0,0.35);
  display: flex; flex-direction: column; transition: all 0.3s;
  animation: fadeUp 0.5s ease-out both;
}
.card:hover { border-color: rgba(255,255,255,0.12); }
@keyframes fadeUp {
  from { opacity: 0; transform: translateY(14px); }
  to { opacity: 1; transform: translateY(0); }
}
.card .kicker {
  display: flex; gap: 10px; align-items: center;
  color: __MUTED__; font-size: 0.72rem; letter-spacing: 0.12em;
  text-transform: uppercase; font-weight: 700; margin-bottom: 6px;
  font-family: 'JetBrains Mono', monospace;
}
.card .kicker .tag {
  background: var(--accent, __HI__); color: __BG__;
  padding: 3px 8px; border-radius: 6px;
  font-family: 'JetBrains Mono', monospace;
  font-size: 0.68rem; font-weight: 700; letter-spacing: 0.05em;
}
.card .cap {
  margin: 10px 2px 0; color: __SOFT__; font-size: 0.84rem;
  line-height: 1.55; padding-top: 12px; border-top: 1px solid __EDGE__;
}
.tab { display: none; }
.tab.on { display: block; animation: fadeIn 0.35s ease-out; }
@keyframes fadeIn { from { opacity: 0; } to { opacity: 1; } }
.foot {
  margin-top: 40px; padding-top: 20px; border-top: 1px solid __EDGE__;
  color: __MUTED__; font-size: 0.8rem; line-height: 1.7; max-width: 1000px;
}
.foot b { color: __INK__; }
.modebar { background: transparent !important; }
.modebar-btn path { fill: __MUTED__ !important; }
.modebar-btn:hover path { fill: __CYAN__ !important; }
@media (max-width: 1000px) {
  .grid { grid-template-columns: 1fr; }
  .hero { grid-template-columns: 1fr; }
  h1 { font-size: 1.8rem; }
  .wrap { padding: 24px 16px 40px; }
}
"""

JS = """
function showTab(i) {
  document.querySelectorAll('.tab').forEach((t, j) => t.classList.toggle('on', j === i));
  document.querySelectorAll('.tabbtn').forEach((b, j) => b.classList.toggle('on', j === i));
  setTimeout(() => {
    document.querySelectorAll('#tab' + i + ' .js-plotly-plot').forEach(g => {
      if (window.Plotly) Plotly.Plots.resize(g);
    });
  }, 40);
  history.replaceState(null, '', '#tab' + (i + 1));
}
function animateCounters() {
  document.querySelectorAll('.kpi .n').forEach(el => {
    const target = parseFloat(el.dataset.target || '0');
    const duration = 900;
    const start = performance.now();
    function tick(now) {
      const elapsed = now - start;
      const progress = Math.min(1, elapsed / duration);
      const ease = 1 - Math.pow(1 - progress, 3);
      el.textContent = (target * ease).toFixed(1);
      if (progress < 1) requestAnimationFrame(tick);
    }
    requestAnimationFrame(tick);
  });
}
window.addEventListener('load', () => {
  animateCounters();
  const m = /tab(\\d)/.exec(location.hash);
  if (m) showTab(Math.min(5, Math.max(0, +m[1] - 1)));
});
"""

HTML_TEMPLATE = """<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>__TITLE__ — Neon</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=Space+Grotesk:wght@500;600;700&family=JetBrains+Mono:wght@500;700&display=swap" rel="stylesheet">
<style>__CSS__</style>
</head>
<body>
<div class="wrap">
  <div class="eyebrow">DA2402 · Project 29 · Neon edition v2.1</div>
  <h1>How far can you trust <span class="grad">a weather forecast?</span></h1>
  <p class="sub">
    Forecast error vs station observations in <b>Kolkata, Visakhapatnam and Thiruvananthapuram</b>.
    Open-Meteo GFS vs NOAA GHCN-Daily. Temperature: Jun 2022 – Aug 2025.
    Rain: Jan 2024 – Aug 2025. Error = forecast − station. Monsoon = Jun–Sep.
  </p>
  <div class="tabs">__NAV__</div>
  __HERO__
  __TABS__
  <div class="foot">
    <b>Sources &amp; licences.</b> Forecasts: Open-Meteo Previous Runs API (gfs_seamless), CC BY 4.0.
    Observations: NOAA NCEI GHCN-Daily, CC0 1.0.
    <b>Libraries.</b> pandas, numpy, plotly, pyarrow.
    <b>Read with care.</b> Missing station days are deleted, never filled; grid points are 3–6 km from the stations;
    the 7 leads share the same observations, so they are not independent tests;
    rain forecasts exist only from 20 Jan 2024 and recorded-only rain days lean wet.
  </div>
</div>
<script>__JS__</script>
</body>
</html>
"""

## Build

In [102]:
def build(cdn=False, open_browser=True):
    include_js_first = "cdn" if cdn else True

    tab_html, nav = [], []
    for ti, (name, sub, accent, plots) in enumerate(TAB_SPEC):
        cards = []
        for pi, (title, fn) in enumerate(plots, 1):
            fig, cap = safe(fn, title)
            div = fig.to_html(
                full_html=False,
                include_plotlyjs=include_js_first,
                div_id=f"p{ti}_{pi}",
                config={"displaylogo": False, "responsive": True,
                        "modeBarButtonsToRemove": ["lasso2d", "select2d"]})
            include_js_first = False
            cards.append(
                f'<div class="card" style="--accent:{accent}; animation-delay:{(pi-1)*0.06}s">'
                f'<div class="kicker"><span class="tag">{ti+1}.{pi}</span>{title}</div>'
                f'{div}<p class="cap">{cap}</p></div>')
        tab_html.append(
            f'<div class="tab{" on" if ti==0 else ""}" id="tab{ti}" style="--accent:{accent}">'
            f'<div class="tabhead"><h2>{name}</h2><p>{sub}</p></div>'
            f'<div class="grid">{"".join(cards)}</div></div>')
        nav.append(
            f'<button class="tabbtn{" on" if ti==0 else ""}" onclick="showTab({ti})" '
            f'style="--accent:{accent}; --accent-glow:{accent}66">'
            f'<span class="num">{ti+1:02d}</span>{name}</button>')

    hero_html = (
        '<div class="hero">'
        f'<div class="kpi big" style="--kpi-accent:{HI}">'
        '<div class="label">Overall trust horizon</div>'
        f'<div class="value"><span class="n" data-target="{trust_all:.1f}">0.0</span><span class="unit">days</span></div>'
        '<div class="note">Shorter of the temperature and rain horizon at default standards</div>'
        '</div>'
        f'<div class="kpi" style="--kpi-accent:{CYAN}">'
        '<div class="label">Temperature</div>'
        f'<div class="value"><span class="n" data-target="{trust_t:.1f}">0.0</span><span class="unit">days</span></div>'
        f'<div class="note">Error stays within {TEMP_THR:.0f} °C</div></div>'
        f'<div class="kpi" style="--kpi-accent:{PINK}">'
        '<div class="label">Rain</div>'
        f'<div class="value"><span class="n" data-target="{trust_r:.1f}">0.0</span><span class="unit">days</span></div>'
        f'<div class="note">≥ {RAIN_THR:.0f}% of rain events correctly forecast</div></div>'
        '</div>')

    css = (CSS.replace("__BG__", BG).replace("__CARD__", CARD).replace("__EDGE__", EDGE)
              .replace("__INK__", INK).replace("__MUTED__", MUTED).replace("__SOFT__", SOFT)
              .replace("__GRID__", GRID).replace("__HI__", HI).replace("__CYAN__", CYAN)
              .replace("__PINK__", PINK))

    html = (HTML_TEMPLATE
            .replace("__TITLE__", "How far can you trust a weather forecast?")
            .replace("__CSS__", css)
            .replace("__NAV__", "".join(nav))
            .replace("__HERO__", hero_html)
            .replace("__TABS__", "".join(tab_html))
            .replace("__JS__", JS))

    with open(OUT_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"wrote {OUT_HTML} ({os.path.getsize(OUT_HTML)/1e6:.1f} MB)")
    if open_browser:
        webbrowser.open(pathlib.Path(OUT_HTML).as_uri())

## PREVIEW HELPERS
Plain-Python helpers to look at one tab (or one plot) inline without writing the HTML.

In [103]:
def preview_plot(tab_index, plot_index):
    """Render one plot (0-based tab and plot index) and print its caption."""
    name, _, _, plots = TAB_SPEC[tab_index]
    title, fn = plots[plot_index]
    fig, cap = safe(fn, title)
    fig.show()
    print(cap)


def preview_tab(tab_index):
    """Render every plot in one tab (0-based index)."""
    name, sub, _, plots = TAB_SPEC[tab_index]
    print(f"== {name}: {sub} ==")
    for pi in range(len(plots)):
        preview_plot(tab_index, pi)

## RUN
Writes `Forecast_Dashboard.html` and opens it in your default browser. Use `open_browser=False` to skip opening, `cdn=True` for a smaller file that loads Plotly from the web.

In [104]:
build(cdn=False, open_browser=True)

wrote c:\Users\Hari Kaeilash G\OneDrive\DCV_project_final\Forecast_Dashboard.html (7.0 MB)
